# שיעור 17 - יצירת סוכני AI מקומיים עם Foundry Local ו-Qwen

במחברת זו תבנה **עוזר הנדסי מקומי** שרץ כולו על תחנת העבודה שלך. בשום שלב לא נעשה שימוש באינג'קט ענן. העוזר יכול:

1. **להפעיל כלים** דרך קריאת פונקציות Qwen דרך Foundry Local.
2. **לרשום ולקרוא קבצים** בתוך תיקיית פרויקט מבודדת.
3. **לנתח קוד** עם מדדים פשוטים.
4. **לחפש תיעוד** באמצעות RAG מקומי (Chroma).
5. **לשתמש בשרת MCP מקומי** (דלג בעדינות אם לא מוגדר).

קוד הסוכן נראה כמעט זהה לשיעורי ענן — רק נקודת הקצה של הלקוח עוברת מהענן ל-`localhost`.


## הגדרה

לפני הפעלת פנקס זה:

1. **התקן את Microsoft Foundry Local** (ראה את [התיעוד](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) למערכת ההפעלה שלך).
2. **הורד והפעל מודל Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. התקן את חבילות הפייתון למטה.

הכל פועל מקומית. מכונה עם ~8 גיגהבייט RAM הינה מינימום ריאלי.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. התחבר ל-Foundry Local

`FoundryLocalManager` מוריד את הדגם במידת הצורך, מפעיל את השירות המקומי, ומספק לנו **נקודת קצה תואמת OpenAI**. לאחר מכן אנו מפנים אליה את ערכת הפיתוח הסטנדרטית של OpenAI. מפתח ה-API הוא מציין מקומי — אין מעורבות של אישור ענן.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. כלים מקומיים (פעולות קבצים מבודדות)

אנו יוצרים פרויקט דוגמה קטן בדיסק, ואז מגדירים כלים המוגבלים לשורש הפרויקט הזה. בדיקת הסביבה המבודדת חשובה אפילו באופן מקומי: כלי שקורא נתיבים אקראיים פועל בהרשאות המשתמש שלך ויכול לגעת בכל דבר שאתה יכול.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG מקומי עם Chroma

אנו משבצים קבוצה קטנה של קטעי תיעוד לאוסף מקומי של Chroma. Chroma פועל בתהליך ושומר וקטורים בדיסק — ללא שרת, ללא ענן. כלי `search_docs` מחלץ את הקטעים הרלוונטיים ביותר לשאילתה.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. לולאת קריאת הכלים

עכשיו אנו רושמים את הכלים אצל המודל באמצעות סכמת הכלים של OpenAI ומבצעים את לולאת קריאת הכלים הסטנדרטית — המודל מבקש כלי, אנו מבצעים אותו באופן מקומי, מזינים את התוצאה חזרה, וחוזרים על הפעולה עד שהמודל מפיק תשובה סופית. קריאת הפונקציות האמינה של Qwen היא מה שהופכת את זה לעבודה במכשיר.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP מקומי (אופציונלי)

MCP הוא פרוטוקול תעבורה, לא שירות ענן — שרת MCP יכול לפעול כתהליך מקומי דרך `stdio`. התא שמתחת מציג כיצד להתחבר לשרת MCP מקומי אם יש לך אחד מוגדר (לדוגמה שרת מערכת קבצים). הוא מדלג בצורה חלקה כאשר `LOCAL_MCP_COMMAND` אינו מוגדר, כך שהמחברת עדיין רצה מקצה לקצה בלעדיו.

הערת אבטחה: שרת MCP מקומי פועל בהרשאות המשתמש שלך. הגבל אותו לתיקיית פרויקט ואמת את הפלט שלו לפני פעולה על פיו.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## סיכום

בנית עוזר הנדסי שרץ כולו במחשב שלך:

- **Foundry Local** סיפקה מודל **Qwen** מאחורי נקודת קצה התואמת ל-OpenAI — כך שקוד הסוכן תואם את השיעורים בענן.
- **כלים במבודד** נתנו לסוכן גישה לקבצים וניתוח קוד מבלי לצאת מספריית הפרויקט.
- **Chroma** סיפקה **RAG מקומי** על התיעוד.
- **Local MCP** הראתה כיצד ניתן להשתמש מחדש במערכת MCP במצב לא מקוון.

בשום שלב לא נעשה שימוש באינפרנציה בענן.

### אתגר

הרחב את הסוכן המקומי כדי:

1. **לעבוד עם מספר שרתי MCP** — לחבר שרת מערכת קבצים ושרת git ולאפשר לסוכן לבחור ביניהם.
2. **להשתמש בזיכרון מקומי** — לשמור היסטוריית שיחה קצרה בדיסק כדי שהעוזר יזכור סבבים קודמים גם לאחר השבתות ועצירות פנקס.
3. **לתמוך בתיאום בין סוכנים מקומיים מרובים** — להוסיף סוכן מקומי שני (למשל מבקר) ולאפשר לשניים לשתף פעולה במשימה.

בשיעור הבא תלמד כיצד לאבטח סוכני AI המופעלים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
